# T27 — Generate full corpus (100–300 trajectories)

Upload this notebook to Kaggle (Create > Notebook > File > Upload Notebook), turn on an
accelerator (GPU T4 x2), and run the cells **in order, top to bottom**, the first time.

**Resuming in a later session:** you do not need to re-run every cell. Only run:
- Cell 2 (GPU check) — harmless, quick sanity check
- Cell 3 (clone/pull) — always, to make sure you're on the latest frozen code
- Cell 5 (env vars) — always, a fresh kernel has none of these set
- Cell 7 (start Ollama) — always, a fresh kernel has no Ollama process running
- Cell 9 (the batch command) — always, as-is, no edits — it resumes and picks its own seed automatically
- Cell 10 / Cell 11 (progress check / download) — whenever you want a status check or a zip

Skip cell 4 (dependency install), cell 4b (ALFWorld data download + base_config.yaml) and
cell 8 (model pull) on a resume **only if** `/kaggle/working` still has them from before (same
session, just restarted the kernel) — if this is a brand new session/machine, run everything
from the top.

## Cell 2 — confirm GPU is attached
Run this first, every session. If it errors or shows no GPU, fix that in Kaggle's
Settings panel (Accelerator > GPU T4 x2) before continuing — everything after this depends
on a GPU being present for Ollama.

In [ ]:
!nvidia-smi

## Cell 3 — clone or pull the repo, verify the commit
Run this every session (first run of the day, and every resume). It clones fresh the first
time, or pulls if the checkout already exists. **Check the printed commit/tag output before
going further** — if `react_v3.1` is not reachable, something didn't pull correctly and you
are not running the frozen, verified code.

In [ ]:
import os

repo_dir = "/kaggle/working/tripwire-agent-monitor"
if not os.path.isdir(repo_dir):
    !git clone -q https://github.com/ManahilAyaz/tripwire-agent-monitor.git {repo_dir}

os.chdir(repo_dir)
!git checkout -q feature/w1-react-v3-prompting 2>/dev/null; git checkout -q main
!git pull -q
print("HEAD:")
!git log -1 --oneline
print("\nIs react_v3.1 reachable from HEAD? (should print a commit line, not be empty)")
!git merge-base --is-ancestor react_v3.1 HEAD && git log -1 --oneline react_v3.1 || echo 'NOT REACHABLE - stop and check before continuing'

## Cell 4 — create a Python 3.10 venv (via uv) and install the project (first time only)
**Skip this on a resume within the same session's disk** — it's slow and `/kaggle/working`
survives a kernel restart within one session, so re-installing is wasted time. Run it on a
genuinely new session/machine. It's safe to run again anyway (idempotent), just slower.

**Why a venv, and why `uv` specifically:** ALFWorld/TextWorld's grammar engine uses `eval()`
in a way that breaks on Python 3.13 (Kaggle's base image) — `NameError: name 'r' is not
defined` deep inside `textworld/envs/pddl/textgen/__init__.py`. `docs/setup.md` already pins
**Python 3.10** for exactly this reason. `uv venv --python 3.10` downloads an isolated Python
3.10 build itself, regardless of what Kaggle's base image ships, and doesn't depend on the
base image's `ensurepip` (which is separately broken here too). Every later cell that runs
Python code against this project uses `/kaggle/working/venv/bin/...`, not plain `python`.

In [ ]:
import os

venv_dir = "/kaggle/working/venv"
venv_python = f"{venv_dir}/bin/python"

!pip install -q uv

if not os.path.isfile(venv_python):
    # --clear: overwrite a broken/partial venv left over from an interrupted session
    # instead of uv's interactive y/n prompt, which hangs forever in a notebook cell.
    !uv venv --python 3.10 {venv_dir} --clear

assert os.path.isfile(venv_python), f"venv creation failed - {venv_python} still missing"

!uv pip install --python {venv_python} -q -r /kaggle/working/tripwire-agent-monitor/requirements.txt
!uv pip install --python {venv_python} -q -e /kaggle/working/tripwire-agent-monitor
!uv pip install --python {venv_python} -q alfworld
print("Python 3.10 venv ready at", venv_dir)
!{venv_python} --version

## Cell 4b — download ALFWorld's game data + fetch base_config.yaml (first time only)
**Skip this on a resume within the same session's disk** — `alfworld-download` pulls real game
data and can be slow; re-running it when `/kaggle/working/alfworld_data` already has it is
wasted time. Run it on a genuinely new session/machine.

`base_config.yaml` is not part of our repo — it ships inside ALFWorld's own repo
(`docs/setup.md`'s Colab instructions: `git clone ... alfworld` then `cp
alfworld/configs/base_config.yaml .`), so it has to be fetched separately and copied into
`tripwire-agent-monitor/`'s root, which is where `scripts/run_batch.py --config
base_config.yaml` expects to find it.

In [ ]:
import os
import shutil

os.environ["ALFWORLD_DATA"] = "/kaggle/working/alfworld_data"
os.makedirs(os.environ["ALFWORLD_DATA"], exist_ok=True)

if not os.listdir(os.environ["ALFWORLD_DATA"]):
    !/kaggle/working/venv/bin/alfworld-download

alfworld_src = "/kaggle/working/alfworld_src"
if not os.path.isdir(alfworld_src):
    !git clone -q https://github.com/alfworld/alfworld {alfworld_src}

shutil.copy(
    f"{alfworld_src}/configs/base_config.yaml",
    "/kaggle/working/tripwire-agent-monitor/base_config.yaml",
)
print("base_config.yaml in place; ALFWORLD_DATA has", len(os.listdir(os.environ["ALFWORLD_DATA"])), "entries")

## (Resume only) restore episodes from a previous Save Version
`/kaggle/working` does **not** survive a brand-new session on its own — only a kernel
restart *within* the same session does. The only way to get `data/raw/full` back after a real
session loss is if you clicked **Save Version** (use **Quick Save** / "save without
re-running", not "Save & Run All" — that would restart cell 9 from scratch) at some point
while those episodes existed.

To restore into a new session: click **Add Input → Your Work → this notebook**, pick the
version you saved, then run the cell below. Skip this on the very first run of a fresh corpus,
or if you never saved a version — in that case there is nothing to restore and `--seed 0`
in cell 9 is correct.


In [ ]:
import glob
import os
import shutil

os.chdir("/kaggle/working/tripwire-agent-monitor")
found = glob.glob("/kaggle/input/**/data/raw/full", recursive=True)
if found:
    shutil.copytree(found[0], "data/raw/full", dirs_exist_ok=True)
    print(f"Restored {len(os.listdir('data/raw/full'))} file(s) from {found[0]}")
else:
    print("Nothing found under /kaggle/input/ to restore.")
    print("Either this is a fresh start, or you haven't done Add Input -> Your Work ->")
    print("this notebook yet (see the markdown cell above).")


## Cell 5 — environment variables
Run this every session — a fresh kernel has none of these set, even if `/kaggle/working` is
untouched. Points `make_groq_llm` at the local Ollama server instead of Groq's API (Ollama
ignores the API key, so any placeholder string works), and points Ollama's own model storage
at `/kaggle/working` so the downloaded model weights survive a kernel restart instead of
re-downloading every time.

In [ ]:
import os

os.environ["LLM_BASE_URL"] = "http://localhost:11434/v1"
os.environ["LLM_API_KEY"] = "ollama-local-placeholder"
os.environ["OLLAMA_MODELS"] = "/kaggle/working/ollama_models"
os.environ["ALFWORLD_DATA"] = "/kaggle/working/alfworld_data"
os.makedirs(os.environ["OLLAMA_MODELS"], exist_ok=True)
os.makedirs(os.environ["ALFWORLD_DATA"], exist_ok=True)
print("env vars set")

## Cell 6 — install Ollama (first time only)
Skip if `ollama` is already on PATH (check with `!which ollama` — if it prints a path, skip
this cell).

In [ ]:
!which ollama || (apt-get update -qq && apt-get install -y -qq zstd pciutils && curl -fsSL https://ollama.com/install.sh | sh)

## Cell 7 — start the Ollama server
Run this every session. If Ollama was already running (e.g. you're just re-running after a
cell hang, not a full restart), this is harmless — the curl check at the end confirms it's
actually responding either way. **If the curl check fails or hangs, this is the exact failure
mode that happened repeatedly before** — re-run this cell once; if it still fails, use
Kaggle's Restart-and-clear-outputs (not just Cancel) and re-run from cell 5.

In [ ]:
import subprocess
import time

subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    start_new_session=True,
)
time.sleep(5)
!curl -s http://localhost:11434/api/tags && echo "\nOllama is up."

## Cell 8 — pull the model (first time only, or if the model dir is empty)
Skip this on a resume if `OLLAMA_MODELS` already has the model cached from a previous run in
this session — Ollama itself also skips the download if it detects the model already present,
so it is safe (just slower) to run this again if you're unsure.

In [ ]:
!ollama pull llama3.1:8b

## Cell 9 — run the batch (the long-running cell)
Run this every session, exactly as-is — no editing needed between sessions. Targets the full
T27 range in one command (`--n 300`). `run_batch.py` is resumable by episode-id file, so this
automatically skips whatever's already finished in `data/raw/full/` and continues from the
next episode.

**The seed is computed automatically, not hardcoded**, from how many episodes are already
finished in `data/raw/full/` — session 1 starts at `--seed 0` (0 done so far); if session 1
finishes, say, 80 episodes before the time limit, session 2 automatically computes `--seed
80`; session 3 picks up from wherever that leaves off, and so on. This removes the need to
check anything or decide a seed yourself between sessions — see the last markdown cell for
exactly what this does and doesn't guarantee.

**This cell launches the batch in the background and returns immediately** instead of
blocking — a Jupyter kernel only executes one cell at a time, so a blocking call here would
make cells 10 and 11 below genuinely unrunnable (not just "safe in theory") until the whole
300-episode run finished, which defeats the point of checking progress or downloading
mid-run. Its full output goes to `/kaggle/working/batch_run.log` instead of this cell's
output. The batch itself still runs until it hits 300 episodes, the Kaggle session's time
limit forces a restart, or 3 episodes in a row error out (Ollama crash) and it stops itself
early — any of those is fine, your progress is saved per-episode as it goes.


In [ ]:
import json
import subprocess
from pathlib import Path

out_dir = Path("data/raw/full")
done_outcomes = {"success", "failure", "max_steps"}
done_count = 0
if out_dir.exists():
    for f in out_dir.glob("ep_*.jsonl"):
        try:
            lines = f.read_text(encoding="utf-8").splitlines()
        except OSError:
            continue
        for line in lines:
            rec = json.loads(line)
            if rec.get("type") == "episode_summary" and rec.get("outcome") in done_outcomes:
                done_count += 1
                break

seed = done_count  # session 1: 0 done -> seed 0. session 2: N done -> seed N. etc.
print(f"{done_count} episode(s) already done in {out_dir} -> using --seed {seed}")

batch_log = open("/kaggle/working/batch_run.log", "a")
batch_proc = subprocess.Popen(
    [
        "/kaggle/working/venv/bin/python", "scripts/run_batch.py",
        "--n", "300", "--config", "base_config.yaml",
        "--agent-config", "configs/agent_ollama.yaml",
        "--out-dir", "data/raw/full", "--seed", str(seed),
    ],
    stdout=batch_log, stderr=subprocess.STDOUT,
)
print(f"Batch started in background (PID {batch_proc.pid}).")
print("Full output: /kaggle/working/batch_run.log")
print("The kernel is free now -> run cell 10 or 11 any time, as many times as you want,")
print("without waiting for this to finish.")


## Cell 10 — progress check (run this any time, including while cell 9 is still running)
Counts finished episodes and their outcomes so far, without needing to wait for the whole
batch to finish. Also reports whether the background batch process is still alive.


In [ ]:
import json
from pathlib import Path
from collections import Counter

out_dir = Path("/kaggle/working/tripwire-agent-monitor/data/raw/full")
files = sorted(out_dir.glob("ep_*.jsonl")) if out_dir.exists() else []
outcomes = Counter()
task_ids = []
for f in files:
    for line in f.read_text(encoding="utf-8").splitlines():
        rec = json.loads(line)
        if rec.get("type") == "episode_summary":
            outcomes[rec["outcome"]] += 1
            task_ids.append(rec.get("task_id"))
print(f"{len(files)} episode file(s) found")
print("outcomes:", dict(outcomes))
print("unique task_ids so far:", len(set(task_ids)), "/", len(task_ids))

try:
    print("batch process still running:", batch_proc.poll() is None)
except NameError:
    print("(batch_proc not in this kernel's memory - rerun cell 9 if needed, or check")
    print(" /kaggle/working/batch_run.log directly)")


## Cell 11 — zip and download whatever's done so far
Run this any time, including while cell 9 is still running in the background — it only reads
`data/raw/full`, it never touches the batch process. Re-run periodically during a long run (not
just once at the end) so you have a recent download if the session gets cut off before you
expect it to.


In [ ]:
import os
import shutil
from IPython.display import FileLink, display

os.chdir("/kaggle/working/tripwire-agent-monitor")
shutil.make_archive("/kaggle/working/corpus_full", "zip", "data/raw/full")
print("Episodes zipped:", len(os.listdir("data/raw/full")))
os.chdir("/kaggle/working")
display(FileLink("corpus_full.zip"))

---
## Note: why episodes could repeat a task across a restart (fixed)

Earlier versions of this notebook guessed this was Python `random`/`numpy.random` reseeding
-> wrong, verified against ALFWorld/TextWorld's actual source. The real cause: TextWorld's
own game sampler (`TextworldBatchGymEnv`, what `env.reset()` calls into) already samples
*without* replacement from a shuffled cycle over every game file - but that cycle's seed
(`1234`) is hardcoded inside TextWorld itself, unrelated to anything `run_batch.py` sets.
Every time a Kaggle session restart forced a brand-new `env` to be built, that cycle reset
to position 0 of the *same* fixed shuffle - so the first episodes after a restart could
redraw a game an earlier run (this one, or even the pilot's) had already used. That's what
produced the 11 excluded episodes documented in `data/raw/full/EXCLUDED.md`.

Fixed in `src/tripwire/runner/batch_runner.py::sequential_game_order` (wired into
`scripts/run_batch.py`, used automatically by cell 9 - no notebook changes needed): it owns
its own shuffled game order and resume cursor, persisted to disk next to the episodes
themselves, instead of trusting TextWorld's restart-unsafe internal cycle. Cell 10's `unique
task_ids so far` count is still there if you want to sanity-check it, but a future run
shouldn't ever report fewer unique task_ids than episodes.


---
## (Optional, once you've reached your target count) replace duplicate-task_id episodes

Random sampling with replacement means a handful of episodes can land on the same
ALFWorld `task_id` (see the note below on why). If you want every episode's task to be
unique, run this **only after the batch has stopped** (check cell 10 says `batch process
still running: False`, or that you've reached your target episode count) - deleting a
file while `run_batch.py` might still be writing it is the one case that's not safe.

It deletes only the *extra* copies (keeping the first episode for each `task_id`), then
you rerun cell 9: `run_batch.py` skips every file that still exists and only regenerates
the deleted slots, each with a freshly sampled task. Re-run this cell + cell 9 again if a
new collision happens to land on the same `task_id` as something else (rare, but possible
for the same reason).


In [ ]:
import json
from pathlib import Path

out_dir = Path("/kaggle/working/tripwire-agent-monitor/data/raw/full")
files = sorted(out_dir.glob("ep_*.jsonl"))

seen_task_ids = set()
dup_files = []
for f in files:
    for line in f.read_text(encoding="utf-8").splitlines():
        rec = json.loads(line)
        if rec.get("type") == "episode_summary":
            tid = rec.get("task_id")
            if tid in seen_task_ids:
                dup_files.append(f)
            else:
                seen_task_ids.add(tid)
            break

print(f"{len(dup_files)} duplicate-task_id episode(s) found:")
for f in dup_files:
    print(" ", f.name)

if dup_files:
    for f in dup_files:
        f.unlink()
    print(f"Deleted {len(dup_files)} file(s). Rerun cell 9 now to regenerate them with fresh tasks.")
else:
    print("No duplicates found - nothing to do.")
